# ALPR — Colab Runner
**One-time setup:** Run cells 1–4 once per session.
**On code updates:** Just run Cell 2 (`git pull`) then Cell 5 to restart Streamlit.

> Make sure you have selected **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ── Cell 2: Clone or pull latest code from GitHub ─────────────────
# First time:  clones the repo
# Every update: pulls latest changes

import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ALPR.git'  # <-- change this
REPO_DIR = '/content/ALPR'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

print('Done — code is up to date')

In [ ]:
# ── Cell 3: Install dependencies (skip if already installed this session) ──
!pip install -q -r /content/ALPR/requirements.txt
print('Dependencies installed')

In [ ]:
# ── Cell 4: Install tunnel (Cloudflare — no bandwidth limit, no account needed) ──
!pip install -q pycloudflared

from pycloudflared import try_cloudflare
print('Cloudflare tunnel ready — will be started in Cell 5')

In [ ]:
# ── Cell 5: Launch Gradio app + Cloudflare tunnel ─────────────────
import subprocess, time, urllib.request

# Kill any previous process
!pkill -f gradio_app || true
!pkill -f streamlit  || true
time.sleep(2)

# Start Gradio in background
log = open('/content/gradio.log', 'w')
proc = subprocess.Popen(
    ['python', '/content/ALPR/gradio_app.py'],
    stdout=log, stderr=log
)

# Wait until Gradio is accepting connections (up to 90s — models take time to load)
print('Waiting for Gradio to start (loading models...)', end='')
for _ in range(90):
    try:
        urllib.request.urlopen('http://localhost:7860', timeout=1)
        print(' ✅ ready')
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    print('\n❌ Gradio did not start — check /content/gradio.log')
    !tail -40 /content/gradio.log
    raise RuntimeError('Gradio failed to start')

# Open Cloudflare tunnel
from pycloudflared import try_cloudflare
url = try_cloudflare(port=7860, verbose=False)
print('=' * 60)
print('  Your app is live at:')
print(f'  {url}')
print('=' * 60)
print('To update code: git pull in Cell 2, then re-run this cell')

In [ ]:
# ── Cell 6 (optional): Upload a video to Colab for testing ────────
from google.colab import files
uploaded = files.upload()   # pick a video from your computer
for fname in uploaded:
    print(f'Uploaded: /content/{fname}')
    print('Use this path in the video app sidebar')

In [ ]:
# ── Cell 7 (optional): Mount Google Drive ─────────────────────────
# If your videos are already on Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted — files are at /content/drive/MyDrive/')